# files

> Files and cells over the gateway's contents and cells APIs

In [ ]:
#| default_exp files

[Rustygate](https://github.com/AnswerDotAI/rustygate) provides a Jupyter-style `/api/contents` API for files and directories and `/api/cells` for editing cells inside notebooks.

Use `JupyAsyncFilesClient` to address files by path. `JupyAsyncCellsClient` works with one notebook's cells. A kernel client bound to that notebook receives `cell_ops` change messages on its `cells` channel. `apply_ops` updates a local list of cells from those messages. Attach `JmsgQueues` if you want to read the broadcasts with `get_jmsg`.

In [ ]:
#| export
import json
from base64 import b64encode, b64decode
from fastcore.basics import patch, patch_to
from fastcore.meta import delegates
from fasttransport.errors import APIError
from jupyasyncclient.core import KernelApi

In [ ]:
import tempfile
from pathlib import Path
from fastcore.test import test_eq, expect_fail
from rustygate.tools import start_gateway

## The files client

Both clients inherit `KernelApi`'s base URL, authentication, and HTTP transport. You can supply an HTTP client or let the transport create one for each request. Writes replace the addressed content; the last accepted edit wins.

In [ ]:
#| export
class JupyAsyncFilesClient(KernelApi):
    "Files and directories over the gateway's contents API."

The file methods call standard contents operations from the bundled OpenAPI specification. Gateway-specific options pass through fastspec's `query_` and `body_` arguments. fastspec leaves out a query argument whose value is `None`. HTTP errors propagate as `APIError`.

The helpers separate query parameters from body fields. For example, `get(path)` requests a model and `put(path, type='directory')` requests directory creation. `patch` passes its body through `body_` because a rename has two paths: the URL's source path and the body's destination `path`.

Use `get` to read a contents model, `put` to write, and `post` to copy through the contents API:

In [ ]:
#| export
@patch
async def get(self:JupyAsyncFilesClient, path='', **kwargs):
    "The model at `path`; `kwargs` become query parameters, e.g. `fields`."
    if not path: return await self.api.contents.get_root(query_=kwargs)
    return await self.api.contents.get_path(path=path, query_=kwargs)

@patch
async def put(self:JupyAsyncFilesClient, path, overwrite=None, unique=None, parents=None, **kwargs):
    "PUT to the contents API; `kwargs` are its JSON body."
    return await self.api.contents.put_path(path=path, query_=dict(overwrite=overwrite, unique=unique, parents=parents), body_=kwargs)

@patch
async def post(self:JupyAsyncFilesClient, path, overwrite=None, unique=None, **kwargs):
    "POST to the contents API; `kwargs` are its JSON body."
    return await self.api.contents.post_path(path=path, query_=dict(overwrite=overwrite, unique=unique), body_=kwargs)

For `patch`, pass the source path positionally. You can then use `path=` for the destination in the body. `delete` removes a file or an empty directory:

In [ ]:
#| export
@patch_to(JupyAsyncFilesClient)
async def patch(self, path, /, **kwargs):
    "PATCH with `kwargs` as the JSON body; `path` is positional-only, freeing the name for the body."
    return await self.api.contents.patch_path(path=path, body_=kwargs)

@patch
async def delete(self:JupyAsyncFilesClient, path):
    "Delete a file or an empty directory."
    return await self.api.contents.delete_path(path=path)

In [ ]:
#| export
@patch
async def write(self:JupyAsyncFilesClient, path, content, unique=False, overwrite=True):
    "Write text or base64-encoded bytes and return the model. `unique` chooses a free `name_n.ext`; `overwrite=False` requires a new file."
    c,f = (b64encode(content).decode(),'base64') if isinstance(content, bytes) else (content,'text')
    return await self.put(path, overwrite=overwrite, unique=unique or None, content=c, format=f)

@patch
async def read(self:JupyAsyncFilesClient, path):
    "A file's contents: `str` for text, `bytes` for binary."
    m = await self.get(path, fields='content')
    return b64decode(m['content']) if m['format']=='base64' else m['content']

@patch
async def listing(self:JupyAsyncFilesClient, path=''):
    "The entries of directory `path`."
    return (await self.get(path))['content']


Let's start a local rustygate process with a temporary directory as its files root. All the following writes stay in that directory:

In [ ]:
root = Path(tempfile.mkdtemp())
g = start_gateway(('rustygate', '--root', root))
fc = JupyAsyncFilesClient(g.url)
m = await fc.write('notes.txt', 'hello')
m

```python
{ 'mtime': 1789266815.6835597,
  'name': 'notes.txt',
  'path': 'notes.txt',
  'size': 5,
  'type': 'file',
  'writable': True}
```

`write` returns the file model. Directory listings return those same stat fields without reading file contents:

In [ ]:
test_eq(await fc.read('notes.txt'), 'hello')
entry = next(e for e in await fc.listing() if e['name']=='notes.txt')
test_eq(entry, m)
entry

```python
{ 'mtime': 1789266815.6835597,
  'name': 'notes.txt',
  'path': 'notes.txt',
  'size': 5,
  'type': 'file',
  'writable': True}
```

Writing an existing file replaces its contents:

In [ ]:
await fc.write('notes.txt', 'hello2')
test_eq(await fc.read('notes.txt'), 'hello2')
await fc.read('notes.txt')

'hello2'

Pass `overwrite=False` to require a new file. An existing path raises `APIError` with status 409 and keeps its content:

In [ ]:
await fc.write('fresh.txt', 'made', overwrite=False)
with expect_fail(APIError): await fc.write('fresh.txt', 'again', overwrite=False)
test_eq(await fc.read('fresh.txt'), 'made')

`write` base64-encodes bytes for transport. `read` returns text when the stored bytes are valid UTF-8, otherwise bytes. Here the binary data round-trips unchanged:

In [ ]:
raw = bytes(range(256))
await fc.write('blob.bin', raw)
back = await fc.read('blob.bin')
test_eq(back, raw)
len(back)

256

`mkdir`, `rename`, and `copy` wrap the contents methods. Use `parents=True` to create missing directory ancestors.

A rename never overwrites an existing destination: it raises `APIError` with status 409. To replace a destination, delete it explicitly first:


In [ ]:
#| export
@patch
async def mkdir(self:JupyAsyncFilesClient, path, parents=False):
    "Create directory `path`; `parents` creates missing ancestors like `mkdir -p`."
    return await self.put(path, parents=parents or None, type='directory')

@patch
async def rename(self:JupyAsyncFilesClient, path, to):
    "Rename and return the new model. An existing destination raises APIError (409); it is never overwritten."
    return await self.patch(path, path=to)

@patch
async def copy(self:JupyAsyncFilesClient, src, to, unique=False, overwrite=True):
    "Copy and return the new model. `unique` chooses a free `name_n.ext`; `overwrite=False` requires a new file."
    return await self.post(to, overwrite=overwrite, unique=unique or None, copy_from=src)

In [ ]:
await fc.mkdir('sub')
await fc.mkdir('deep/a/b', parents=True)
await fc.copy('notes.txt', 'sub/notes.txt')
await fc.rename('sub/notes.txt', 'sub/renamed.txt')
with expect_fail(APIError): await fc.rename('sub/renamed.txt', 'notes.txt')
test_eq([e['name'] for e in await fc.listing('sub')], ['renamed.txt'])
await fc.delete('sub/renamed.txt')
await fc.delete('sub')
await fc.delete('blob.bin')
[e['name'] for e in await fc.listing()]

['deep', 'fresh.txt', 'notes.txt']

Pass `unique=True` to write or copy under the first free `name_n.ext` instead of overwriting. The response contains the final name and path:

In [ ]:
m3 = await fc.write('notes.txt', 'variant', unique=True)
test_eq(m3['name'], 'notes_1.txt')
m4 = await fc.copy('notes.txt', 'notes.txt', unique=True)
test_eq(m4['name'], 'notes_2.txt')
test_eq(await fc.read('notes_2.txt'), await fc.read('notes.txt'))
m4['path']

'notes_2.txt'

Use `edit` to change a JSON file. It reads the content, calls `f` to modify the parsed object in place, and writes the result once. Like any file write, it replaces intervening changes:

In [ ]:
#| export
@patch
async def edit(self:JupyAsyncFilesClient, path, f):
    "Read the JSON file at `path`, apply `f` to the parsed object, and write it back."
    o = json.loads(await self.read(path))
    f(o)
    return await self.write(path, json.dumps(o, sort_keys=True, indent=1))

In [ ]:
await fc.write('cfg.json', json.dumps(dict(a=1)))
def _bump(o): o['a'] += 1
await fc.edit('cfg.json', _bump)
test_eq(json.loads(await fc.read('cfg.json'))['a'], 2)

## The cells client

Construct `JupyAsyncCellsClient` with a notebook path or `kernel_id` and an explicit full Rustygate `spec`. The bundled public spec excludes the cells operations. Kernel-addressed cell operations follow the current binding through rename and restart. Inherited file methods still take explicit paths.

`view` returns the selected `cells` with their notebook `path`. Pass `fields='meta'` to include notebook `metadata`. `cells` returns the selected cell list.

`await client[id]` returns one cell; a tuple of ids returns a list.

In [ ]:
#| export
class JupyAsyncCellsClient(JupyAsyncFilesClient):
    "One notebook's cells over the gateway's cells API."
    def __init__(self, base_url, path=None, token=None, headers=None, timeout=30, http_client=None, verify=True, kernel_id=None, *, spec):
        super().__init__(base_url, token=token, headers=headers, timeout=timeout, http_client=http_client, verify=verify, spec=spec)
        if (path is None)==(kernel_id is None): raise ValueError('Pass path or kernel_id')
        self.path,self.kernel_id = None if path is None else str(path),kernel_id

    def __getitem__(self, ids): return self._lookup(ids)

In [ ]:
#| export
def _cs(v):
    "A comma-separated str from a str, an int, an iterable, or None"
    if v is None or isinstance(v, str): return v
    return ','.join(map(str, v)) if hasattr(v, '__iter__') else str(v)

@patch
async def view(
    self:JupyAsyncCellsClient,
    ids=None, # Cell ids to keep: comma-separated str, or a list
    idx=None, # Cell positions to keep: comma-separated str, or a list of ints, 0-based, negative from the end; unions with `ids`
    section=None, # Cell id selecting a heading and its descendants; a non-heading selects itself
    ancestors=None, # Cell id whose enclosing headings to select, outermost first, excluding the cell itself
    before=None, # Cell id: select every cell before it, excluding it
    after=None, # Cell id: select every cell after it, excluding it
    q=None, # Keep only cells whose source matches this regex (multiline, smart-case)
    cell_type=None, # Keep only cells of this type: 'code', 'markdown', or 'raw'
    meta=None, # Keep only cells whose metadata contains this dict as a recursive subset; a None value means "key present"
    meta_not=None, # Drop cells whose metadata contains this dict as a recursive subset
    limit=None, # Keep at most this many cells after filtering
    max_tokens=None, # Then keep the newest cells whose approximate token counts total at most this many
    context=None, # Also return this many neighbours either side of each kept cell; `matched` in the result names the true matches
    fields=None, # Comma-separated cell fields; '*' includes attachments, 'meta' adds notebook metadata; None omits attachments
    compare=None, # Compare inputs with this Git ref; returns its resolved commit and computed cell change fields
):
    "The selected cells with their notebook path, requested metadata, matched ids and comparison commit."
    return await (self.api.cells.get_cells if self.kernel_id is None else self.api.kernels.get_kernel_cells)(
        **(dict(path=self.path) if self.kernel_id is None else dict(kid=self.kernel_id)),
        ids=_cs(ids), idx=_cs(idx), section=section, ancestors=ancestors, before=before, after=after, q=q, cell_type=cell_type,
        meta=None if meta is None else json.dumps(meta), meta_not=None if meta_not is None else json.dumps(meta_not),
        limit=limit, max_tokens=max_tokens, context=context, fields=fields, compare=compare)

@patch
@delegates(JupyAsyncCellsClient.view)
async def cells(self:JupyAsyncCellsClient, ids=None, **kwargs):
    "The selected cells in document order."
    return (await self.view(ids=ids, **kwargs))['cells']

`apply` submits cell operations in order. `command` submits a semantic message operation, such as `bookmark`, `split`, or `merge`, with its fields as keyword arguments. Both return the added ids and use the client's path or kernel binding. `_lookup` implements bracket notation:

In [ ]:
#| export
@patch
async def apply(self:JupyAsyncCellsClient, ops):
    "Apply `ops` in order, returning ids of added cells."
    m = await (self.api.cells.post_cells if self.kernel_id is None else self.api.kernels.post_kernel_cells)(
        **(dict(path=self.path) if self.kernel_id is None else dict(kid=self.kernel_id)), ops=ops)
    return m['added_ids']

@patch
async def command(self:JupyAsyncCellsClient, op, **kwargs):
    "Apply a semantic message command, returning ids of added cells."
    m = await self.api.messages.post_messages(op=op,
        **(dict(path=self.path) if self.kernel_id is None else dict(kid=self.kernel_id)), body_=kwargs)
    return m['added_ids']

@patch
async def _lookup(self:JupyAsyncCellsClient, ids):
    one = isinstance(ids, str)
    want = [ids] if one else list(ids)
    got = await self.cells(ids=want)
    if len(got)!=len(want): raise KeyError(', '.join(i for i in want if i not in {c['id'] for c in got}))
    return got[0] if one else got

Your application supplies `base_url` and a parsed full Rustygate specification as `full_spec`. For example, Solveit's `rg_full_spec()` reads its packaged full snapshot. Cells operations are not available from the bundled public specification:

In [ ]:
#| eval: false
nb = JupyAsyncCellsClient(base_url, 'analysis.ipynb', spec=full_spec)
await nb.cells(ids=['intro', 'result'], fields='id,source')
await nb.apply([dict(op='update', id='result', source='6*7')])
await nb.command('bookmark', id='intro', n=1)

['aaa1', 'bbb2']

## Applying ops

`apply_ops` updates a list of cell dictionaries in place from broadcast operations. A client can keep its local view current without fetching the notebook after each change.

The gateway converts richer request operations, such as cell metadata `merge`, into add/update/delete broadcasts containing their results; `move` broadcasts carry IDs and placement and preserve the existing cell objects. Notebook-level `meta` leaves the cell list unchanged. Handle file-level `rename` and attachment events separately; passing them to this helper raises `ValueError`.

`output_changed` is event intent, not a cell field. Metadata/source snapshots preserve locally committed output; an intentional replacement applies even when its output list is empty. Execution counts remain ordinary canonical fields.

In [ ]:
#| export
def _update_cell(cell, fields):
    cell.update({k:v for k,v in fields.items() if k not in ('op','id','output_changed') and (k!='outputs' or fields.get('output_changed'))})
    if cell['cell_type']=='code':
        cell.setdefault('outputs', [])
        cell.setdefault('execution_count', None)
    else:
        cell.pop('outputs', None)
        cell.pop('execution_count', None)

def _insert_cells(cells, rows, op):
    ids = [c['id'] for c in cells]
    anchor = op.get('after') or op.get('before')
    at = ids.index(anchor) + bool(op.get('after')) if anchor in ids else len(cells)
    cells[at:at] = rows

In [ ]:
#| export
def apply_ops(cells, ops):
    "Apply cell broadcast operations in order to `cells` in place and return that list."
    for o in ops:
        ids = [c['id'] for c in cells]
        op = o['op']
        if op=='move':
            by_id = {c['id']:c for c in cells}
            rows = [by_id[i] for i in o['ids'] if i in by_id]
            cells[:] = [c for c in cells if c['id'] not in o['ids']]
            _insert_cells(cells, rows, o)
        elif op=='update' and o['id'] in ids: _update_cell(cells[ids.index(o['id'])], o)
        elif op in ('add','update'):
            c = dict(o['cell']) if op=='add' else {k:v for k,v in o.items() if k not in ('op','output_changed')}
            if c.get('id') in ids: _update_cell(cells[ids.index(c['id'])], c | dict(output_changed=o.get('output_changed', False)))
            else:
                c.setdefault('cell_type', 'code')
                c.setdefault('metadata', {})
                if c['cell_type']=='code':
                    c.setdefault('outputs', [])
                    c.setdefault('execution_count', None)
                _insert_cells(cells, [c], o)
        elif op=='delete':
            if o['id'] in ids: del cells[ids.index(o['id'])]
        elif op=='meta': pass
        else: raise ValueError(f"unhandled op: {op}")
    return cells

Here we add a code cell before an existing cell, delete another cell, and turn the new cell into a Markdown heading. The type change removes its execution count and outputs:

In [ ]:
view = [dict(id='aaa1', cell_type='code', source='1+1', metadata={}, outputs=[], execution_count=1),
    dict(id='bbb2', cell_type='markdown', source='# old', metadata={})]
ops = [dict(op='add', cell=dict(id='ccc3', cell_type='code', source='3'), before='aaa1'),
    dict(op='delete', id='bbb2'), dict(op='update', id='ccc3', cell_type='markdown', source='# Calculation')]
apply_ops(view, ops)
test_eq([c['id'] for c in view], ['ccc3', 'aaa1'])
assert 'outputs' not in view[0] and 'execution_count' not in view[0]
[(c['id'], c['cell_type']) for c in view]

[('ccc3', 'markdown'), ('aaa1', 'code'), ('97270c84', 'code')]

An add with an existing id updates that cell in place, ignoring the anchor. A missing anchor appends, and deleting a missing cell has no effect. Here the duplicate add turns `ccc3` back into code with empty outputs and a null execution count:

In [ ]:
ops = [dict(op='delete', id='missing'), dict(op='meta', metadata=dict(topic='calculation')),
    dict(op='add', cell=dict(id='ccc3', cell_type='code', source='3b'), after='missing'),
    dict(op='add', cell=dict(id='ddd4', source='4'), after='missing')]
apply_ops(view, ops)
test_eq([c['id'] for c in view], ['ccc3', 'aaa1', 'ddd4'])
test_eq((view[0]['outputs'], view[0]['execution_count']), ([], None))
test_eq(view[-1], dict(id='ddd4', source='4', cell_type='code', metadata={}, outputs=[], execution_count=None))
kept = view[-1]
apply_ops(view, [dict(op='move', ids=['ddd4', 'ccc3'], before='aaa1')])
test_eq([c['id'] for c in view], ['ddd4', 'ccc3', 'aaa1'])
assert view[0] is kept
kept['outputs'] = [dict(output_type='stream', name='stdout', text='live')]
apply_ops(view, [dict(op='update', id='ddd4', source='new', execution_count=2, outputs=[])])
test_eq((kept['outputs'][0]['text'], kept['execution_count']), ('live', 2))
apply_ops(view, [dict(op='update', id='ddd4', outputs=[], output_changed=True)])
test_eq(kept['outputs'], [])
assert 'output_changed' not in kept


In [ ]:
#| hide
g.stop()

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()